#DPO

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets trl wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.4/517.4 kB 44.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig
import bitsandbytes as bnb
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata

HF_TOKEN =  userdata.get('HF_TOKEN')
WB_TOKEN =  userdata.get('WANDB_API_KEY')

In [ ]:
import wandb
if WB_TOKEN:
    wandb.login(key=WB_TOKEN)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: garodisk (garodisk-university-of-cincinnati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
model_name = "teknium/OpenHermes-2.5-Mistral-7B"
new_model = "teknium/OpenHermes-2.5-Mistral-7B-DPO"

In [ ]:
#for wandb experiemnt tracking

project_name = "mistral-dpo-orca"
run_name     = "openhermes-orca-dpo"

In [ ]:
# Load raw preference dataset
raw_dataset = load_dataset("Intel/orca_dpo_pairs")['train']

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

In [ ]:
raw_dataset

Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 12859
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

In [ ]:
# For reference, columns in Intel/orca_dpo_pairs:
# ['system', 'question', 'chosen', 'rejected']

In [ ]:
def chatml_format(example):
    # Format system prompt (optional)
    if example.get("system") and len(example["system"]) > 0:
        sys_msg = [{"role": "system", "content": example["system"]}]
        system_part = tokenizer.apply_chat_template(
            sys_msg,
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        system_part = ""

    # Format user question
    user_msg = [{"role": "user", "content": example["question"]}]
    prompt_part = tokenizer.apply_chat_template(
        user_msg,
        tokenize=False,
        add_generation_prompt=True,   # adds assistant prefix
    )

    # Preferred & rejected answers (DPO-style)
    chosen  = example["chosen"]   + "<|im_end|>\n"
    rejected = example["rejected"] + "<|im_end|>\n"

    return {
        "prompt": system_part + prompt_part,
        "chosen": chosen,
        "rejected": rejected,
    }


In [ ]:
original_cols = raw_dataset.column_names

dataset = raw_dataset.map(
    chatml_format,
    remove_columns=original_cols,
    desc="Formatting ChatML + DPO columns",
)

# (Optional) subsample for quick experiments
# dataset = dataset.shuffle(seed=42).select(range(5000))

print(dataset[0])

Formatting ChatML + DPO columns:   0%|          | 0/12859 [00:00<?, ? examples/s]

{'chosen': '[\n  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],\n  ["Ajax Youth Academy", "plays at", "Sportpark De Toekomst"]\n]<|im_end|>\n', 'rejected': " Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:\n\n[AFC Ajax (amateurs), hasGround, Sportpark De Toekomst]\n[Ajax Youth Academy, playsAt, Sportpark De Toekomst]\n\nExplanation:\n\n* AFC Ajax (amateurs) is the subject of the first triplet, and hasGround is the predicate that describes the relationship between AFC Ajax (amateurs) and Sportpark De Toekomst.\n* Ajax Youth Academy is the subject of the second triplet, and playsAt is the predicate that describes the relationship between Ajax Youth Academy and Sportpark De Toekomst.\n\nNote that there may be other possible RDF triplets that could be derived from the input sentence, but the above triplets capture the main relationships present in the sentence.<|im_end|>\n", 'prompt': "<|im_start|>user\nYou will be given a definition of a task

In [ ]:
dataset

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 12859
})

In [ ]:
print(dataset[300])

{'chosen': 'Step 1: Identify the text to be transformed to lower case, which is "Together with other industrial sectors, ARBURG was presented as a model of best practice in the plastics industry."\n\nStep 2: Apply the lower case function to the given text.\n\ntogether with other industrial sectors, arburg was presented as a model of best practice in the plastics industry.\n\nStep 3: Provide the transformed text in lower case: "together with other industrial sectors, arburg was presented as a model of best practice in the plastics industry."<|im_end|>\n', 'rejected': ' Sure, I\'d be happy to help! Here\'s the task you\'ve given me:\n\n"Make the following sentence lower case: \'Together with other industrial sectors, ARBURG was presented as a model of best practice in the plastics industry.\'"\n\nTo complete this task, I will follow these steps:\n\nStep 1: Identify the words that need to be changed to lower case.\n\nIn the given sentence, the words that need to be changed to lower case a

In [ ]:
for key, value in dataset[100].items():
    print(f"{key.capitalize()}:\n{value}\n")

Chosen:
This is a positive sentiment. The person might have had some doubts at the beginning, especially about P. Brosnan playing an American Indian, but in the end, they found the film fascinating and very good. They enjoyed the performances of the actors, especially Annie Galipeau, and appreciated the movie's portrayal of native peoples. Even though they thought the plot could have done a better job showing the character's change from a trapper to an animal conservationist, they still highly recommend the film. It's like when you try a new flavor of ice cream, even if you weren't sure about it at first, you end up really liking it and want your friends to try it too!<|im_end|>


Rejected:
 OH MY GOSH! 😱 You watched a movie about Grey Owl? 🐦 That's so cool! 😎 I just loooove movies about real people, don't you? 🤗

So, you thought P. Brosnan as an American Indian might be a bad choice? 🤔 Well, I can see why you might think that, but let me tell you, he did a GREAT job! 👏 He's like, supe

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
# LoRA configuration (modern TRL/PEFT style)
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

In [ ]:
# Policy model to fine-tune
policy_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
policy_model.config.use_cache = False

# Reference model (frozen - no need to load actually)
# ref_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     token=HF_TOKEN,
# )
# ref_model.config.use_cache = False

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 5. TRAINING CONFIG (DPOConfig) + TRAINER
# ============================================================
training_config = DPOConfig(
    # core training params
    output_dir=new_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    max_steps=200,             # bump this up later if stable
    warmup_steps=50,
    optim="paged_adamw_32bit",

    # logging / tracking
    logging_steps=1,
    report_to="wandb",
    run_name=run_name,

    # mixed precision
    bf16=True,

    # DPO-specific
    beta=0.1,
    max_prompt_length=1024,
    max_length=1536,

    # saving
    save_strategy="no",        # we’ll save manually later
)



In [ ]:
dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,
    args=training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# ============================================================
# 6. TRAIN WITH DPO
# ============================================================
dpo_trainer.train()

# Optionally log final metrics
metrics = dpo_trainer.state.log_history[-1] if dpo_trainer.state.log_history else {}
print("Final training metrics:", metrics)

Extracting prompt in train dataset:   0%|          | 0/12859 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/12859 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12859 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32000}.


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
1,0.693100
2,0.693100
3,0.686900
4,0.666400
5,0.642800
6,0.635300
7,0.564700
8,0.548200
9,0.499700
10,0.443200


Final training metrics: {'train_runtime': 3003.4692, 'train_samples_per_second': 1.065, 'train_steps_per_second': 0.067, 'total_flos': 0.0, 'train_loss': 0.03986356771852539, 'epoch': 0.24883359253499224, 'step': 200}


In [ ]:
# ============================================================
# 7. SAVE ADAPTER CHECKPOINT
# ============================================================
import os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_dir = "final_dpo_adapter"
os.makedirs(adapter_dir, exist_ok=True)

# Save just the LoRA adapter + tokenizer
dpo_trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Free quantized models from memory (optional)
del dpo_trainer, policy_model, ref_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:


# ============================================================
# 8. MERGE ADAPTER + BASE INTO FULL FP16 MODEL
# ============================================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,                 # "teknium/OpenHermes-2.5-Mistral-7B"
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load LoRA adapter into the FP16 base and merge
merged = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=False,
)
merged = merged.merge_and_unload()

# Local folder name for the merged model
local_merged_dir = "OpenHermes-2.5-Mistral-7B-DPO"
os.makedirs(local_merged_dir, exist_ok=True)

merged.save_pretrained(local_merged_dir)
tokenizer.save_pretrained(local_merged_dir)

print("Merged model saved locally to:", local_merged_dir)





`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model saved locally to: OpenHermes-2.5-Mistral-7B-DPO


In [ ]:


# ============================================================
# 9. PUSH TO HUGGING FACE HUB (UNDER saketgarodia1)
# ============================================================
hub_model_id = "saketgarodia1/OpenHermes-2.5-Mistral-7B-DPO"

if HF_TOKEN is None:
    print("HF_TOKEN not found – skipping push_to_hub. Set HF_TOKEN env/secret.")
else:
    merged.push_to_hub(
        hub_model_id,
        use_temp_dir=False,
        token=HF_TOKEN,
    )
    tokenizer.push_to_hub(
        hub_model_id,
        use_temp_dir=False,
        token=HF_TOKEN,
    )
    print(f"Pushed {hub_model_id} to the Hub.")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00003.safetensors:   0%|          |  524kB / 5.00GB            

  ...0001-of-00003.safetensors:   1%|          | 33.5MB / 4.94GB            

  ...0003-of-00003.safetensors:   1%|          | 33.5MB / 4.54GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al-7B-DPO/tokenizer.model: 100%|##########|  493kB /  493kB            

Pushed saketgarodia1/OpenHermes-2.5-Mistral-7B-DPO to the Hub.


In [ ]:


# ============================================================
# 10. QUICK INFERENCE TEST (FROM LOCAL OR HUB)
# ============================================================
import transformers

# You can test from the local folder or directly from the Hub:
# test_model_id = local_merged_dir
test_model_id = hub_model_id  # once pushed

tokenizer = AutoTokenizer.from_pretrained(test_model_id, token=HF_TOKEN)
pipe = transformers.pipeline(
    "text-generation",
    model=test_model_id,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a helpful assistant chatbot."},
    {"role": "user", "content": "Explain what a Large Language Model is, in 3 short bullet points."},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

out = pipe(
    prompt,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    max_new_tokens=200,
)

print(out[0]["generated_text"])


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Device set to use cuda:0


<|im_start|>system
You are a helpful assistant chatbot.<|im_end|>
<|im_start|>user
Explain what a Large Language Model is, in 3 short bullet points.<|im_end|>
<|im_start|>assistant
1. Large Language Models (LLMs) are artificial intelligence systems designed to process, understand, and generate human language.
2. They are built using deep learning techniques, primarily neural networks with numerous layers, allowing them to learn from vast amounts of text data.
3. These models can perform various natural language processing tasks, such as language translation, text generation, and question answering, with increasing accuracy as they grow in size and complexity.
